In [3]:
def main(datasources, start_date, end_date):
    """
    BigAlpha submission: Alpha158-style daily factors + multi-level OFI/intraday path model.

    Design:
    1) Compute per-instrument daily Alpha158-style features on continuous price history.
    2) Compute exact daily-pool, five-level OFI and intraday price-response features in SQL.
    3) Make bigalpha_2026_instruments the master panel before ANY same-day rank/label operation.
    4) Train a daily Alpha158 model and a separate microstructure model.
    5) Daily-rank both model predictions inside the official pool and blend them.

    Important universe rule:
    - Per-stock rolling features are computed before filtering to the daily pool so index
      entry/exit does not break a stock's own history.
    - Cross-sectional ranks and labels are computed only after the official pool is the
      master panel, so non-constituents cannot affect the cross-section.
    """
    import time
    import numpy as np
    import pandas as pd
    import dai
    import xgboost as xgb
    import structlog

    logger = structlog.get_logger()

    TRAIN_START = "2019-01-01 00:00:00"
    TRAIN_END = "2024-12-31 23:59:59"

    # Maximum Alpha158 window is 60 trading observations. 120 calendar days gives
    # enough warm-up while staying within the competition's stated lookback limit.
    LOOKBACK_DAYS = 120

    # Blend weights are deliberately explicit and easy to tune.
    ALPHA_WEIGHT = 0.70
    OFI_WEIGHT = 0.30

    EPS = 1e-12
    ALPHA_WINDOWS = (5, 10, 20, 30, 60)

    def get_pool(sd, ed):
        pool = dai.query(
            "SELECT date, instrument FROM bigalpha_2026_instruments",
            filters={"date": [sd, ed]},
        ).df()
        pool["date"] = pd.to_datetime(pool["date"].astype(str))
        pool["instrument"] = pool["instrument"].astype(str)
        return (
            pool[["date", "instrument"]]
            .drop_duplicates(["date", "instrument"])
            .sort_values(["date", "instrument"])
            .reset_index(drop=True)
        )

    def query_daily_price(bar1m_table, sd, ed):
        query_start = pd.to_datetime(sd) - pd.Timedelta(days=LOOKBACK_DAYS)
        sql = f"""
        SELECT
            date_trunc('day', date)::DATE AS trading_day,
            instrument,
            ARG_MIN(open, date) AS open,
            MAX(high) AS high,
            MIN(low) AS low,
            ARG_MAX(close, date) AS close,
            SUM(CAST(volume AS DOUBLE)) AS volume,
            SUM(CAST(amount AS DOUBLE)) AS amount
        FROM {bar1m_table}
        GROUP BY trading_day, instrument
        ORDER BY instrument, trading_day
        """
        price = dai.query(
            sql,
            filters={"date": [query_start, ed]},
            compression=True,
        ).df().rename(columns={"trading_day": "date"})

        price["date"] = pd.to_datetime(price["date"].astype(str))
        price["instrument"] = price["instrument"].astype(str)
        for c in ["open", "high", "low", "close", "volume", "amount"]:
            price[c] = pd.to_numeric(price[c], errors="coerce")
            price[c] = price[c].replace([np.inf, -np.inf], np.nan)

        price["vwap"] = price["amount"] / (price["volume"].abs() + EPS)
        price.loc[~np.isfinite(price["vwap"]), "vwap"] = np.nan
        return price.sort_values(["instrument", "date"]).reset_index(drop=True)

    def query_intraday_ofi(bar1m_table, sd, ed):
        """
        Five-level OFI plus intraday path/price-response aggregation.

        Unlike the daily Alpha158 history, these features are entirely within-day,
        so joining the exact daily official pool in SQL is both safe and efficient.
        """
        query_start = pd.to_datetime(sd) - pd.Timedelta(days=LOOKBACK_DAYS)

        # IMPORTANT: depth/volume fields can be INT32 in the source table.
        # Cast them to DOUBLE before *any* addition/subtraction. Casting only the
        # final result is too late because DuckDB evaluates intermediate INT32
        # additions first, which can overflow for large queue sizes.
        level_select = []
        for level in range(1, 6):
            level_select.extend([
                f"CAST(b.bid_price{level} AS DOUBLE) AS bid_price{level}",
                f"CAST(b.ask_price{level} AS DOUBLE) AS ask_price{level}",
                f"COALESCE(CAST(b.bid_volume{level} AS DOUBLE), 0.0) AS bid_volume{level}",
                f"COALESCE(CAST(b.ask_volume{level} AS DOUBLE), 0.0) AS ask_volume{level}",
            ])
        level_select_sql = ",\n                ".join(level_select)

        lag_select = []
        for level in range(1, 6):
            for side in ("bid", "ask"):
                lag_select.append(
                    f"LAG({side}_price{level}) OVER "
                    f"(PARTITION BY trading_day, instrument ORDER BY date) "
                    f"AS prev_{side}_price{level}"
                )
                lag_select.append(
                    f"LAG({side}_volume{level}) OVER "
                    f"(PARTITION BY trading_day, instrument ORDER BY date) "
                    f"AS prev_{side}_volume{level}"
                )
        lag_select_sql = ",\n                ".join(lag_select)

        ofi_raw_exprs = []
        ofi_norm_exprs = []
        for level in range(1, 6):
            raw_name = f"ofi_l{level}_raw"
            raw_expr = f"""
                CASE
                    WHEN prev_bid_price{level} IS NULL OR prev_ask_price{level} IS NULL
                         OR bid_price{level} <= 0 OR ask_price{level} <= 0
                         OR prev_bid_price{level} <= 0 OR prev_ask_price{level} <= 0
                    THEN 0.0
                    ELSE
                        CASE WHEN bid_price{level} >= prev_bid_price{level}
                             THEN bid_volume{level} ELSE 0.0 END
                        - CASE WHEN bid_price{level} <= prev_bid_price{level}
                               THEN prev_bid_volume{level} ELSE 0.0 END
                        - CASE WHEN ask_price{level} <= prev_ask_price{level}
                               THEN ask_volume{level} ELSE 0.0 END
                        + CASE WHEN ask_price{level} >= prev_ask_price{level}
                               THEN prev_ask_volume{level} ELSE 0.0 END
                END AS {raw_name}
            """
            ofi_raw_exprs.append(raw_expr.strip())
            ofi_norm_exprs.append(
                f"{raw_name} / ("
                f"bid_volume{level} + ask_volume{level} + "
                f"COALESCE(prev_bid_volume{level}, 0) + "
                f"COALESCE(prev_ask_volume{level}, 0) + 1e-8) "
                f"AS ofi_l{level}_norm"
            )
        ofi_raw_sql = ",\n                ".join(ofi_raw_exprs)
        ofi_norm_sql = ",\n                ".join(ofi_norm_exprs)

        sql = f"""
        WITH base AS (
            SELECT
                b.date,
                b.instrument,
                date_trunc('day', b.date)::DATE AS trading_day,
                EXTRACT(HOUR FROM b.date) * 60 + EXTRACT(MINUTE FROM b.date)
                    AS minute_of_day,
                COALESCE(CAST(b.volume AS DOUBLE), 0.0) AS minute_volume,
                {level_select_sql}
            FROM {bar1m_table} b
            INNER JOIN bigalpha_2026_instruments p
                ON date_trunc('day', b.date)::DATE = CAST(p.date AS DATE)
               AND b.instrument = p.instrument
            WHERE b.date BETWEEN '{query_start}' AND '{ed}'
              AND b.bid_price1 > 0
              AND b.ask_price1 > 0
              AND b.ask_price1 >= b.bid_price1
              AND COALESCE(CAST(b.bid_volume1 AS DOUBLE), 0.0)
                  + COALESCE(CAST(b.ask_volume1 AS DOUBLE), 0.0) > 0.0
        ),

        lagged AS (
            SELECT
                *,
                (bid_price1 + ask_price1) / 2.0 AS mid_price,
                LAG((bid_price1 + ask_price1) / 2.0) OVER (
                    PARTITION BY trading_day, instrument ORDER BY date
                ) AS prev_mid_price,
                {lag_select_sql}
            FROM base
        ),

        with_raw_ofi AS (
            SELECT
                *,
                {ofi_raw_sql}
            FROM lagged
        ),

        minute_core AS (
            SELECT
                *,
                {ofi_norm_sql},
                (ask_price1 - bid_price1) / (mid_price + 1e-8)
                    AS relative_spread,
                (
                    ask_price1 * bid_volume1 + bid_price1 * ask_volume1
                ) / (bid_volume1 + ask_volume1 + 1e-8)
                    AS microprice,
                (
                    (bid_volume1 + bid_volume2 + bid_volume3 + bid_volume4 + bid_volume5)
                    - (ask_volume1 + ask_volume2 + ask_volume3 + ask_volume4 + ask_volume5)
                ) / (
                    bid_volume1 + bid_volume2 + bid_volume3 + bid_volume4 + bid_volume5
                    + ask_volume1 + ask_volume2 + ask_volume3 + ask_volume4 + ask_volume5
                    + 1e-8
                ) AS depth_imbalance,
                (
                    bid_volume1 + bid_volume2 + bid_volume3 + bid_volume4 + bid_volume5
                    + ask_volume1 + ask_volume2 + ask_volume3 + ask_volume4 + ask_volume5
                ) AS total_depth,
                (bid_volume1 + ask_volume1) / (
                    bid_volume1 + bid_volume2 + bid_volume3 + bid_volume4 + bid_volume5
                    + ask_volume1 + ask_volume2 + ask_volume3 + ask_volume4 + ask_volume5
                    + 1e-8
                ) AS l1_depth_share,
                CASE
                    WHEN prev_mid_price > 0 AND mid_price > 0
                    THEN LN(mid_price / prev_mid_price)
                    ELSE 0.0
                END AS minute_log_return
            FROM with_raw_ofi
        ),

        integrated AS (
            SELECT
                *,
                (
                    1.0 * ofi_l1_norm
                    + 0.8 * ofi_l2_norm
                    + 0.6 * ofi_l3_norm
                    + 0.4 * ofi_l4_norm
                    + 0.2 * ofi_l5_norm
                ) / 3.0 AS ofi_integrated,
                (microprice - mid_price) / (ask_price1 - bid_price1 + 1e-8)
                    AS microprice_deviation
            FROM minute_core
        ),

        path AS (
            SELECT
                *,
                LAG(ofi_integrated) OVER (
                    PARTITION BY trading_day, instrument ORDER BY date
                ) AS prev_ofi_integrated,
                EXP(-0.08 * GREATEST(899 - minute_of_day, 0))
                    AS close_decay_weight
            FROM integrated
        ),

        daily AS (
            SELECT
                trading_day,
                instrument,
                COUNT(*) AS n_minutes,
                SUM(minute_volume) AS intraday_volume,

                AVG(relative_spread) AS relative_spread_mean,
                STDDEV(relative_spread) AS relative_spread_std,
                MAX(relative_spread) AS relative_spread_max,
                AVG(total_depth) AS total_depth_mean,
                AVG(l1_depth_share) AS l1_depth_share_mean,
                AVG(depth_imbalance) AS depth_imbalance_mean,
                STDDEV(depth_imbalance) AS depth_imbalance_std,
                AVG(CASE WHEN minute_of_day BETWEEN 885 AND 899
                         THEN depth_imbalance END) AS close15_depth_imbalance_mean,

                AVG(microprice_deviation) AS microprice_deviation_mean,
                STDDEV(microprice_deviation) AS microprice_deviation_std,
                AVG(CASE WHEN minute_of_day BETWEEN 885 AND 899
                         THEN microprice_deviation END) AS close15_microprice_deviation_mean,

                SUM(ofi_l1_norm) AS ofi_l1_sum,
                SUM(ofi_l2_norm) AS ofi_l2_sum,
                SUM(ofi_l3_norm) AS ofi_l3_sum,
                SUM(ofi_l4_norm) AS ofi_l4_sum,
                SUM(ofi_l5_norm) AS ofi_l5_sum,

                SUM(ofi_integrated) AS ofi_integrated_sum,
                AVG(ofi_integrated) AS ofi_integrated_mean,
                STDDEV(ofi_integrated) AS ofi_integrated_std,
                SUM(ABS(ofi_integrated)) AS ofi_integrated_abs_sum,
                AVG(CASE WHEN ofi_integrated > 0 THEN 1.0 ELSE 0.0 END)
                    AS ofi_positive_ratio,
                AVG(CASE
                        WHEN prev_ofi_integrated IS NOT NULL
                         AND ofi_integrated * prev_ofi_integrated < 0
                        THEN 1.0 ELSE 0.0
                    END) AS ofi_sign_change_ratio,
                CORR(ofi_integrated, prev_ofi_integrated) AS ofi_autocorr,
                CORR(ofi_integrated, minute_log_return) AS ofi_return_corr,
                COVAR_POP(ofi_integrated, minute_log_return)
                    / (VAR_POP(ofi_integrated) + 1e-8) AS ofi_price_impact,

                SUM(minute_log_return) AS full_log_return,
                SQRT(SUM(minute_log_return * minute_log_return)) AS realized_vol,
                SUM(ABS(minute_log_return)) AS total_abs_return,
                ABS(SUM(minute_log_return))
                    / (SUM(ABS(minute_log_return)) + 1e-8) AS path_efficiency,
                AVG(CASE WHEN ofi_integrated > 0
                         THEN minute_log_return END) AS positive_ofi_return_mean,
                AVG(CASE WHEN ofi_integrated < 0
                         THEN minute_log_return END) AS negative_ofi_return_mean,
                AVG(CASE
                        WHEN ofi_integrated * minute_log_return > 0 THEN 1.0
                        WHEN ofi_integrated * minute_log_return < 0 THEN -1.0
                        ELSE 0.0
                    END) AS ofi_return_alignment,

                SUM(CASE WHEN minute_of_day BETWEEN 570 AND 599
                         THEN ofi_integrated ELSE 0.0 END) AS open30_ofi_sum,
                SUM(CASE WHEN minute_of_day BETWEEN 840 AND 899
                         THEN ofi_integrated ELSE 0.0 END) AS close60_ofi_sum,
                SUM(CASE WHEN minute_of_day BETWEEN 870 AND 899
                         THEN ofi_integrated ELSE 0.0 END) AS close30_ofi_sum,
                SUM(CASE WHEN minute_of_day BETWEEN 885 AND 899
                         THEN ofi_integrated ELSE 0.0 END) AS close15_ofi_sum,
                SUM(CASE WHEN minute_of_day BETWEEN 895 AND 899
                         THEN ofi_integrated ELSE 0.0 END) AS close5_ofi_sum,

                SUM(CASE WHEN minute_of_day BETWEEN 570 AND 599
                         THEN ABS(ofi_integrated) ELSE 0.0 END) AS open30_ofi_abs_sum,
                SUM(CASE WHEN minute_of_day BETWEEN 840 AND 899
                         THEN ABS(ofi_integrated) ELSE 0.0 END) AS close60_ofi_abs_sum,
                SUM(CASE WHEN minute_of_day BETWEEN 870 AND 899
                         THEN ABS(ofi_integrated) ELSE 0.0 END) AS close30_ofi_abs_sum,
                SUM(CASE WHEN minute_of_day BETWEEN 885 AND 899
                         THEN ABS(ofi_integrated) ELSE 0.0 END) AS close15_ofi_abs_sum,
                SUM(CASE WHEN minute_of_day BETWEEN 895 AND 899
                         THEN ABS(ofi_integrated) ELSE 0.0 END) AS close5_ofi_abs_sum,

                SUM(CASE WHEN minute_of_day BETWEEN 570 AND 599
                         THEN minute_log_return ELSE 0.0 END) AS open30_log_return,
                SUM(CASE WHEN minute_of_day BETWEEN 840 AND 899
                         THEN minute_log_return ELSE 0.0 END) AS close60_log_return,
                SUM(CASE WHEN minute_of_day BETWEEN 870 AND 899
                         THEN minute_log_return ELSE 0.0 END) AS close30_log_return,
                SUM(CASE WHEN minute_of_day BETWEEN 885 AND 899
                         THEN minute_log_return ELSE 0.0 END) AS close15_log_return,
                SUM(CASE WHEN minute_of_day BETWEEN 895 AND 899
                         THEN minute_log_return ELSE 0.0 END) AS close5_log_return,

                SUM(CASE WHEN minute_of_day BETWEEN 570 AND 599
                         THEN ABS(minute_log_return) ELSE 0.0 END) AS open30_abs_return,
                SUM(CASE WHEN minute_of_day BETWEEN 840 AND 899
                         THEN ABS(minute_log_return) ELSE 0.0 END) AS close60_abs_return,
                SUM(CASE WHEN minute_of_day BETWEEN 870 AND 899
                         THEN ABS(minute_log_return) ELSE 0.0 END) AS close30_abs_return,
                SUM(CASE WHEN minute_of_day BETWEEN 885 AND 899
                         THEN ABS(minute_log_return) ELSE 0.0 END) AS close15_abs_return,
                SUM(CASE WHEN minute_of_day BETWEEN 895 AND 899
                         THEN ABS(minute_log_return) ELSE 0.0 END) AS close5_abs_return,

                SUM(ofi_integrated * close_decay_weight)
                    / (SUM(close_decay_weight) + 1e-8) AS decayed_close_ofi,

                SUM(CASE WHEN minute_of_day BETWEEN 570 AND 599
                         THEN minute_volume ELSE 0 END)
                    / (SUM(minute_volume) + 1e-8) AS volume_open_share,
                SUM(CASE WHEN minute_of_day BETWEEN 870 AND 899
                         THEN minute_volume ELSE 0 END)
                    / (SUM(minute_volume) + 1e-8) AS volume_close_share
            FROM path
            GROUP BY trading_day, instrument
        )

        SELECT * FROM daily
        ORDER BY instrument, trading_day
        """

        intraday = dai.query(
            sql,
            filters={"date": [query_start, ed]},
            compression=True,
        ).df().rename(columns={"trading_day": "date"})

        intraday["date"] = pd.to_datetime(intraday["date"].astype(str))
        intraday["instrument"] = intraday["instrument"].astype(str)
        for c in intraday.columns:
            if c not in ("date", "instrument"):
                intraday[c] = pd.to_numeric(intraday[c], errors="coerce")
        return intraday.sort_values(["instrument", "date"]).reset_index(drop=True)

    def rolling_regression_stats(s, window):
        """Efficient rolling slope, R-squared, and last-point residual."""
        y = pd.to_numeric(s, errors="coerce").astype(float)
        n_obs = len(y)
        positions = np.arange(n_obs, dtype=float)

        sum_y = y.rolling(window, min_periods=window).sum()
        sum_y2 = (y * y).rolling(window, min_periods=window).sum()
        sum_py = (y * positions).rolling(window, min_periods=window).sum()

        start_pos = positions - window + 1.0
        # Local x coordinates are 1..window.
        sum_xy = sum_py - (start_pos - 1.0) * sum_y
        sum_x = window * (window + 1.0) / 2.0
        sum_x2 = window * (window + 1.0) * (2.0 * window + 1.0) / 6.0
        sxx = sum_x2 - (sum_x * sum_x) / window

        slope = (sum_xy - sum_x * sum_y / window) / (sxx + EPS)
        intercept = (sum_y - slope * sum_x) / window
        fitted_last = intercept + slope * window
        residual = y - fitted_last

        sst = sum_y2 - (sum_y * sum_y) / window
        rsquare = (slope * slope * sxx) / (sst + EPS)
        rsquare = rsquare.where(sst > 1e-10).clip(lower=0.0, upper=1.0)
        return slope, rsquare, residual

    def rolling_current_rank(s, window):
        roll = s.rolling(window, min_periods=window)
        try:
            return roll.rank(method="average", pct=True)
        except Exception:
            return roll.apply(
                lambda x: np.mean(x <= x[-1]) if np.isfinite(x[-1]) else np.nan,
                raw=True,
            )

    def compute_alpha158_group(group):
        """Pandas implementation of Qlib's default Alpha158 feature families."""
        g = group.sort_values("date")
        features = {}

        o = g["open"].astype(float)
        h = g["high"].astype(float)
        l = g["low"].astype(float)
        c = g["close"].astype(float)
        v = g["volume"].astype(float)
        vw = g["vwap"].astype(float)

        candle_range = h - l
        greater_oc = pd.concat([o, c], axis=1).max(axis=1)
        lesser_oc = pd.concat([o, c], axis=1).min(axis=1)

        # 9 K-bar features.
        features["A158_KMID"] = (c - o) / (o.abs() + EPS)
        features["A158_KLEN"] = (h - l) / (o.abs() + EPS)
        features["A158_KMID2"] = (c - o) / (candle_range.abs() + EPS)
        features["A158_KUP"] = (h - greater_oc) / (o.abs() + EPS)
        features["A158_KUP2"] = (h - greater_oc) / (candle_range.abs() + EPS)
        features["A158_KLOW"] = (lesser_oc - l) / (o.abs() + EPS)
        features["A158_KLOW2"] = (lesser_oc - l) / (candle_range.abs() + EPS)
        features["A158_KSFT"] = (2.0 * c - h - l) / (o.abs() + EPS)
        features["A158_KSFT2"] = (2.0 * c - h - l) / (candle_range.abs() + EPS)

        # 4 current price-level features used by the default Alpha158 handler.
        features["A158_OPEN0"] = o / (c.abs() + EPS)
        features["A158_HIGH0"] = h / (c.abs() + EPS)
        features["A158_LOW0"] = l / (c.abs() + EPS)
        features["A158_VWAP0"] = vw / (c.abs() + EPS)

        close_ratio_1 = c / (c.shift(1).abs() + EPS)
        volume_ratio_1 = v / (v.shift(1).abs() + EPS)
        price_change = c - c.shift(1)
        volume_change = v - v.shift(1)
        abs_price_change = price_change.abs()
        abs_volume_change = volume_change.abs()
        weighted_move = (close_ratio_1 - 1.0).abs() * v
        log_volume = np.log1p(v.clip(lower=0))
        log_volume_ratio = np.log1p(volume_ratio_1.clip(lower=0))

        up = (price_change > 0).astype(float)
        down = (price_change < 0).astype(float)
        positive_change = price_change.clip(lower=0)
        negative_change = (-price_change).clip(lower=0)
        positive_vchange = volume_change.clip(lower=0)
        negative_vchange = (-volume_change).clip(lower=0)

        for w in ALPHA_WINDOWS:
            roll_c = c.rolling(w, min_periods=w)
            roll_h = h.rolling(w, min_periods=w)
            roll_l = l.rolling(w, min_periods=w)
            roll_v = v.rolling(w, min_periods=w)

            max_h = roll_h.max()
            min_l = roll_l.min()
            slope, rsquare, residual = rolling_regression_stats(c, w)
            idxmax = roll_h.apply(lambda x: float(np.argmax(x) + 1), raw=True)
            idxmin = roll_l.apply(lambda x: float(np.argmin(x) + 1), raw=True)

            features[f"A158_ROC{w}"] = c.shift(w) / (c.abs() + EPS)
            features[f"A158_MA{w}"] = roll_c.mean() / (c.abs() + EPS)
            features[f"A158_STD{w}"] = roll_c.std() / (c.abs() + EPS)
            features[f"A158_BETA{w}"] = slope / (c.abs() + EPS)
            features[f"A158_RSQR{w}"] = rsquare
            features[f"A158_RESI{w}"] = residual / (c.abs() + EPS)
            features[f"A158_MAX{w}"] = max_h / (c.abs() + EPS)
            features[f"A158_MIN{w}"] = min_l / (c.abs() + EPS)
            features[f"A158_QTLU{w}"] = roll_c.quantile(0.8) / (c.abs() + EPS)
            features[f"A158_QTLD{w}"] = roll_c.quantile(0.2) / (c.abs() + EPS)
            features[f"A158_RANK{w}"] = rolling_current_rank(c, w)
            features[f"A158_RSV{w}"] = (c - min_l) / (max_h - min_l + EPS)
            features[f"A158_IMAX{w}"] = idxmax / w
            features[f"A158_IMIN{w}"] = idxmin / w
            features[f"A158_IMXD{w}"] = (idxmax - idxmin) / w
            features[f"A158_CORR{w}"] = roll_c.corr(log_volume)
            features[f"A158_CORD{w}"] = close_ratio_1.rolling(
                w, min_periods=w
            ).corr(log_volume_ratio)

            cntp = up.rolling(w, min_periods=w).mean()
            cntn = down.rolling(w, min_periods=w).mean()
            features[f"A158_CNTP{w}"] = cntp
            features[f"A158_CNTN{w}"] = cntn
            features[f"A158_CNTD{w}"] = cntp - cntn

            abs_change_sum = abs_price_change.rolling(w, min_periods=w).sum()
            sump = positive_change.rolling(w, min_periods=w).sum() / (abs_change_sum + EPS)
            sumn = negative_change.rolling(w, min_periods=w).sum() / (abs_change_sum + EPS)
            features[f"A158_SUMP{w}"] = sump
            features[f"A158_SUMN{w}"] = sumn
            features[f"A158_SUMD{w}"] = sump - sumn

            features[f"A158_VMA{w}"] = roll_v.mean() / (v.abs() + EPS)
            features[f"A158_VSTD{w}"] = roll_v.std() / (v.abs() + EPS)
            features[f"A158_WVMA{w}"] = (
                weighted_move.rolling(w, min_periods=w).std()
                / (weighted_move.rolling(w, min_periods=w).mean().abs() + EPS)
            )

            abs_vchange_sum = abs_volume_change.rolling(w, min_periods=w).sum()
            vsump = positive_vchange.rolling(w, min_periods=w).sum() / (
                abs_vchange_sum + EPS
            )
            vsumn = negative_vchange.rolling(w, min_periods=w).sum() / (
                abs_vchange_sum + EPS
            )
            features[f"A158_VSUMP{w}"] = vsump
            features[f"A158_VSUMN{w}"] = vsumn
            features[f"A158_VSUMD{w}"] = vsump - vsumn

        out = pd.DataFrame(features, index=g.index)
        # Default Alpha158 = 9 K-bar + 4 price + 29 operators * 5 windows.
        if out.shape[1] != 158:
            raise RuntimeError(f"Alpha158 feature count mismatch: {out.shape[1]}")
        return out

    def add_intraday_derived_features(df):
        eps = 1e-8
        df["lob_missing"] = df["ofi_integrated_sum"].isna().astype("float32")
        df["low_coverage"] = (df["n_minutes"].fillna(0) < 180).astype("float32")
        df["log_total_depth_mean"] = np.log1p(
            df["total_depth_mean"].clip(lower=0)
        )

        df["near_minus_deep_ofi"] = (
            (df["ofi_l1_sum"] + df["ofi_l2_sum"]) / 2.0
            - (df["ofi_l4_sum"] + df["ofi_l5_sum"]) / 2.0
        )
        df["ofi_depth_slope"] = df["ofi_l1_sum"] - df["ofi_l5_sum"]
        df["ofi_directional_ratio"] = (
            df["ofi_integrated_sum"] / (df["ofi_integrated_abs_sum"] + eps)
        )
        df["price_directional_ratio"] = (
            df["full_log_return"] / (df["total_abs_return"] + eps)
        )
        df["ofi_price_divergence"] = (
            df["ofi_directional_ratio"] - df["price_directional_ratio"]
        )

        for window_name in ("open30", "close60", "close30", "close15", "close5"):
            ofi_sum = df[f"{window_name}_ofi_sum"]
            ofi_abs = df[f"{window_name}_ofi_abs_sum"]
            ret = df[f"{window_name}_log_return"]
            abs_ret = df[f"{window_name}_abs_return"]
            df[f"{window_name}_ofi_direction"] = ofi_sum / (ofi_abs + eps)
            df[f"{window_name}_price_direction"] = ret / (abs_ret + eps)
            df[f"{window_name}_ofi_price_divergence"] = (
                df[f"{window_name}_ofi_direction"]
                - df[f"{window_name}_price_direction"]
            )
            df[f"{window_name}_path_efficiency"] = ret.abs() / (abs_ret + eps)

        df["late_vs_early_ofi"] = (
            df["close30_ofi_direction"] - df["open30_ofi_direction"]
        )
        df["close15_vs_close60_ofi"] = (
            df["close15_ofi_direction"] - df["close60_ofi_direction"]
        )
        df["close5_ofi_acceleration"] = (
            df["close5_ofi_direction"] - df["close15_ofi_direction"]
        )
        df["late_price_reversal"] = (
            df["close30_log_return"] - df["full_log_return"]
        )
        df["price_response_asymmetry"] = (
            df["positive_ofi_return_mean"]
            + df["negative_ofi_return_mean"]
        )
        df["hidden_liquidity_proxy"] = (
            df["ofi_directional_ratio"].abs()
            * (1.0 - df["path_efficiency"].clip(0.0, 1.0))
        )
        df["close15_hidden_liquidity_proxy"] = (
            df["close15_ofi_direction"].abs()
            * (1.0 - df["close15_path_efficiency"].clip(0.0, 1.0))
        )
        df["ofi_confirmed_return"] = (
            df["ofi_directional_ratio"] * df["full_log_return"]
        )
        df["close15_ofi_confirmed_return"] = (
            df["close15_ofi_direction"] * df["close15_log_return"]
        )
        return df

    def build_features(bar1m_table, sd, ed):
        t0 = time.time()
        logger.info("开始构建 Alpha158 + 多层OFI 特征", start=str(sd), end=str(ed))

        price = query_daily_price(bar1m_table, sd, ed)
        intraday = query_intraday_ofi(bar1m_table, sd, ed)

        # Compute next observed daily return before applying the index pool. This avoids
        # an index exit/re-entry date becoming an artificial "next day".
        price["fwd_ret_1"] = (
            price.groupby("instrument")["close"].shift(-1) / price["close"] - 1.0
        )

        logger.info("计算 Alpha158", rows=len(price), instruments=price["instrument"].nunique())
        alpha_parts = []
        for _, group in price.groupby("instrument", sort=False):
            alpha_parts.append(compute_alpha158_group(group))
        alpha = pd.concat(alpha_parts, axis=0).sort_index()
        alpha_cols = list(alpha.columns)
        price = pd.concat([price, alpha], axis=1)

        df = pd.merge(
            price,
            intraday,
            how="left",
            on=["date", "instrument"],
            validate="one_to_one",
        )
        df = add_intraday_derived_features(df)

        # Model-ready numeric conversion before the pool merge.
        ignore_cols = {"date", "instrument"}
        for col in df.columns:
            if col not in ignore_cols:
                df[col] = pd.to_numeric(df[col], errors="coerce")
                df[col] = df[col].replace([np.inf, -np.inf], np.nan)

        # CRITICAL: official daily pool becomes the master panel before every
        # same-day cross-sectional feature or label transformation.
        query_start = pd.to_datetime(sd) - pd.Timedelta(days=LOOKBACK_DAYS)
        pool = get_pool(query_start, ed)
        rows_before_pool = len(df)
        df = pd.merge(
            pool,
            df,
            how="left",
            on=["date", "instrument"],
            validate="one_to_one",
        ).sort_values(["date", "instrument"]).reset_index(drop=True)

        logger.info(
            "已切换到官方BigAlpha日度股票池",
            rows_before=rows_before_pool,
            pool_rows=len(df),
        )

        # Alpha158 is used in pool-relative percentile-rank form. This is more aligned
        # with the competition's cross-sectional target and avoids doubling the model
        # with both raw and rank copies of all 158 variables.
        alpha_rank_cols = [f"{c}_CSRK" for c in alpha_cols]
        alpha_ranked = (
            df.groupby("date")[alpha_cols]
            .rank(method="average", pct=True)
            .mul(2.0)
            .sub(1.0)
        )
        alpha_ranked.columns = alpha_rank_cols
        df = pd.concat([df, alpha_ranked.astype("float32")], axis=1)

        base_exclude = {
            "date", "instrument", "open", "high", "low", "close", "vwap",
            "volume", "amount", "fwd_ret_1",
        }
        micro_cols = [
            c for c in df.columns
            if c not in base_exclude
            and not c.startswith("A158_")
            and c not in ("lob_missing", "low_coverage")
            and not c.endswith("_CSRK")
        ]

        micro_rank_cols = [f"{c}_CSRK" for c in micro_cols]
        micro_ranked = (
            df.groupby("date")[micro_cols]
            .rank(method="average", pct=True)
            .mul(2.0)
            .sub(1.0)
        )
        micro_ranked.columns = micro_rank_cols
        df = pd.concat([df, micro_ranked.astype("float32")], axis=1)

        # Cross-sectional label over exactly the same official daily pool.
        label_group = df.groupby("date")["fwd_ret_1"]
        df["label"] = (
            df["fwd_ret_1"] - label_group.transform("mean")
        ) / (label_group.transform("std") + 1e-9)

        # Only return the requested interval; warm-up rows were used for rolling features.
        df = df[
            (df["date"] >= pd.to_datetime(sd))
            & (df["date"] <= pd.to_datetime(ed))
        ].reset_index(drop=True)

        logger.info(
            "特征构建完成",
            rows=len(df),
            alpha_features=len(alpha_rank_cols),
            ofi_features=len(micro_rank_cols),
            elapsed=round(time.time() - t0, 2),
        )
        return df, alpha_rank_cols, micro_rank_cols

    # ------------------------- training -------------------------
    logger.info("开始构建训练集", train_start=TRAIN_START, train_end=TRAIN_END)
    train_df, alpha_train_cols, ofi_train_cols = build_features(
        "bigalpha_2026_stock_bar1m", TRAIN_START, TRAIN_END
    )
    train_df["label"] = train_df["label"].replace([np.inf, -np.inf], np.nan)
    train_df = train_df.dropna(subset=["label"]).reset_index(drop=True)

    # Keep quality flags raw because they are binary rather than ordinal signals.
    ofi_model_cols = ofi_train_cols + ["lob_missing", "low_coverage"]

    logger.info(
        "开始训练两个模型",
        samples=len(train_df),
        alpha_features=len(alpha_train_cols),
        ofi_features=len(ofi_model_cols),
    )

    alpha_model = xgb.XGBRegressor(
        n_estimators=550,
        max_depth=3,
        learning_rate=0.025,
        subsample=0.78,
        colsample_bytree=0.72,
        colsample_bynode=0.85,
        min_child_weight=45,
        gamma=0.03,
        reg_lambda=10.0,
        reg_alpha=1.0,
        tree_method="hist",
        max_bin=96,
        n_jobs=-1,
        random_state=42,
        eval_metric="rmse",
    )
    alpha_model.fit(train_df[alpha_train_cols], train_df["label"], verbose=False)

    ofi_model = xgb.XGBRegressor(
        n_estimators=420,
        max_depth=3,
        learning_rate=0.025,
        subsample=0.75,
        colsample_bytree=0.75,
        colsample_bynode=0.85,
        min_child_weight=60,
        gamma=0.05,
        reg_lambda=14.0,
        reg_alpha=1.5,
        tree_method="hist",
        max_bin=64,
        n_jobs=-1,
        random_state=73,
        eval_metric="rmse",
    )
    ofi_model.fit(train_df[ofi_model_cols], train_df["label"], verbose=False)

    # ------------------------- prediction -------------------------
    logger.info("开始构建测试集并预测")
    test_df, alpha_test_cols, ofi_test_cols = build_features(
        datasources["bar1m"], start_date, end_date
    )

    # Defensive schema check: training/test feature names must be identical.
    if alpha_test_cols != alpha_train_cols:
        raise RuntimeError("Alpha158 train/test feature schema mismatch")
    if ofi_test_cols != ofi_train_cols:
        raise RuntimeError("OFI train/test feature schema mismatch")

    test_df["pred_alpha_raw"] = alpha_model.predict(test_df[alpha_train_cols])
    test_df["pred_ofi_raw"] = ofi_model.predict(test_df[ofi_model_cols])

    # Rank each sub-model inside the exact official pool for each date before blending.
    test_df["pred_alpha_rank"] = (
        test_df.groupby("date")["pred_alpha_raw"]
        .rank(method="average", pct=True)
        .mul(2.0)
        .sub(1.0)
    )
    test_df["pred_ofi_rank"] = (
        test_df.groupby("date")["pred_ofi_raw"]
        .rank(method="average", pct=True)
        .mul(2.0)
        .sub(1.0)
    )
    test_df["factor"] = (
        ALPHA_WEIGHT * test_df["pred_alpha_rank"]
        + OFI_WEIGHT * test_df["pred_ofi_rank"]
    )

    result = test_df[["date", "instrument", "factor"]].copy()
    result["factor"] = result["factor"].replace([np.inf, -np.inf], np.nan)
    result = (
        result.dropna(subset=["factor"])
        .drop_duplicates(["date", "instrument"])
        .sort_values(["date", "instrument"])
        .reset_index(drop=True)
    )

    logger.info(
        "因子构建完成",
        rows=len(result),
        alpha_weight=ALPHA_WEIGHT,
        ofi_weight=OFI_WEIGHT,
    )
    return result[["date", "instrument", "factor"]]


if __name__ == "__main__":
    from bigmodule import M
    import dai
    import structlog

    logger = structlog.get_logger()
    datasources = {
        "bar1m": "bigalpha_2026_stock_bar1m",
        "financial": "bigalpha_2026_financial",
    }
    start_date = "2024-01-01 00:00:00"
    end_date = "2024-12-31 23:59:59"

    logger.info(f"计算因子，测试区间：{start_date} ~ {end_date}")
    factor_data = main(datasources, start_date, end_date)

    factor_pool = dai.query(
        "SELECT * FROM bigalpha_2026_factorlib",
        filters={"date": [start_date, end_date]},
    ).df()

    result = M.bigalpha_eval._latest(
        factor_data=factor_data,
        factor_pool=factor_pool,
        process_pools=False,
        show=True,
    )


[2026-08-04 16:02:16] [info     ] 计算因子，测试区间：2024-01-01 00:00:00 ~ 2024-12-31 23:59:59
[2026-08-04 16:02:16] [info     ] 开始构建训练集                        train_end='2024-12-31 23:59:59' train_start='2019-01-01 00:00:00'
[2026-08-04 16:02:16] [info     ] 开始构建 Alpha158 + 多层OFI 特征       end='2024-12-31 23:59:59' start='2019-01-01 00:00:00'


[2026-08-04 16:30:00] [info     ] 计算 Alpha158                    instruments=2252 rows=2916320
[2026-08-04 16:35:03] [info     ] 已切换到官方BigAlpha日度股票池            pool_rows=1535000 rows_before=2916320
